In [0]:
# CommerceLake - Source Data Validation

BASE_PATH = "/Volumes/commercelake/landing/source_data"

BATCH_ID = "batch_001"

SOURCE_CONFIG = {
    "customers": {
        "format": "csv",
        "filename": "customers.csv",
        "expected_count": 1000
    },
    "products": {
        "format": "csv",
        "filename": "products.csv",
        "expected_count": 200
    },
    "orders": {
        "format": "json",
        "filename": "orders.json",
        "expected_count": 5000
    },
    "order_items": {
        "format": "parquet",
        "filename": "order_items.parquet",
        "expected_count": 10055
    },
    "payments": {
        "format": "json",
        "filename": "payments.json",
        "expected_count": 5000
    }
}

print("=" * 60)
print("COMMERCELAKE SOURCE CONFIGURATION")
print("=" * 60)

for entity, config in SOURCE_CONFIG.items():

    path = (
        f"{BASE_PATH}/{entity}/"
        f"{BATCH_ID}/{config['filename']}"
    )

    config["path"] = path

    print(f"\nEntity: {entity}")
    print(f"Format: {config['format']}")
    print(f"Expected records: {config['expected_count']:,}")
    print(f"Source path: {path}")

total_expected = sum(
    config["expected_count"]
    for config in SOURCE_CONFIG.values()
)

print(f"\nTotal source datasets: {len(SOURCE_CONFIG)}")
print(f"Total expected records: {total_expected:,}")

print("\nSource configuration initialized successfully.")

COMMERCELAKE SOURCE CONFIGURATION

Entity: customers
Format: csv
Expected records: 1,000
Source path: /Volumes/commercelake/landing/source_data/customers/batch_001/customers.csv

Entity: products
Format: csv
Expected records: 200
Source path: /Volumes/commercelake/landing/source_data/products/batch_001/products.csv

Entity: orders
Format: json
Expected records: 5,000
Source path: /Volumes/commercelake/landing/source_data/orders/batch_001/orders.json

Entity: order_items
Format: parquet
Expected records: 10,055
Source path: /Volumes/commercelake/landing/source_data/order_items/batch_001/order_items.parquet

Entity: payments
Format: json
Expected records: 5,000
Source path: /Volumes/commercelake/landing/source_data/payments/batch_001/payments.json

Total source datasets: 5
Total expected records: 21,255

Source configuration initialized successfully.


In [0]:
print("=" * 60)
print("SOURCE FILE EXISTENCE VALIDATION")
print("=" * 60)

file_validation_results = []

for entity, config in SOURCE_CONFIG.items():

    directory = f"{BASE_PATH}/{entity}/{BATCH_ID}"

    files = dbutils.fs.ls(directory)

    actual_files = [file.name for file in files]

    expected_file = config["filename"]

    file_exists = expected_file in actual_files

    unexpected_files = [
        name for name in actual_files
        if name != expected_file
    ]

    status = (
        "PASSED"
        if file_exists and not unexpected_files
        else "FAILED"
    )

    file_size = next(
        (file.size for file in files
         if file.name == expected_file),
        0
    )

    file_validation_results.append({
        "entity": entity,
        "filename": expected_file,
        "file_exists": file_exists,
        "file_size_bytes": file_size,
        "unexpected_files": len(unexpected_files),
        "status": status
    })

    print(f"\nEntity: {entity}")
    print(f"Expected file: {expected_file}")
    print(f"File exists: {file_exists}")
    print(f"File size: {file_size:,} bytes")
    print(f"Unexpected files: {unexpected_files}")
    print(f"Status: {status}")


all_files_valid = all(
    result["status"] == "PASSED"
    for result in file_validation_results
)

print("\n" + "=" * 60)

if all_files_valid:
    print("ALL SOURCE FILE VALIDATIONS PASSED")
else:
    raise RuntimeError("Source file validation failed.")

SOURCE FILE EXISTENCE VALIDATION

Entity: customers
Expected file: customers.csv
File exists: True
File size: 171,044 bytes
Unexpected files: []
Status: PASSED

Entity: products
Expected file: products.csv
File exists: True
File size: 27,458 bytes
Unexpected files: []
Status: PASSED

Entity: orders
Expected file: orders.json
File exists: True
File size: 1,685,858 bytes
Unexpected files: []
Status: PASSED

Entity: order_items
Expected file: order_items.parquet
File exists: True
File size: 391,937 bytes
Unexpected files: []
Status: PASSED

Entity: payments
Expected file: payments.json
File exists: True
File size: 2,030,395 bytes
Unexpected files: []
Status: PASSED

ALL SOURCE FILE VALIDATIONS PASSED


In [0]:
print("=" * 60)
print("PYSPARK SOURCE DATA READING")
print("=" * 60)

source_dataframes = {}

for entity, config in SOURCE_CONFIG.items():

    file_format = config["format"]
    file_path = config["path"]

    print(f"\nReading entity: {entity}")
    print(f"File format: {file_format}")

    try:

        if file_format == "csv":

            df = (
                spark.read
                .option("header", "true")
                .option("inferSchema", "false")
                .option("mode", "FAILFAST")
                .csv(file_path)
            )

        elif file_format == "json":

            df = (
                spark.read
                .option("multiLine", "false")
                .option("mode", "FAILFAST")
                .json(file_path)
            )

        elif file_format == "parquet":

            df = spark.read.parquet(file_path)

        else:

            raise ValueError(
                f"Unsupported format: {file_format}"
            )

        source_dataframes[entity] = df

        print(f"Columns: {len(df.columns)}")
        print("DataFrame initialized successfully.")

    except Exception as error:

        print(f"Failed to read {entity}: {error}")
        raise


print("\n" + "=" * 60)

print(f"Total DataFrames initialized: {len(source_dataframes)}")

print("ALL SOURCE DATAFRAMES INITIALIZED SUCCESSFULLY")

PYSPARK SOURCE DATA READING

Reading entity: customers
File format: csv
Columns: 13
DataFrame initialized successfully.

Reading entity: products
File format: csv
Columns: 12
DataFrame initialized successfully.

Reading entity: orders
File format: json
Columns: 11
DataFrame initialized successfully.

Reading entity: order_items
File format: parquet
Columns: 15
DataFrame initialized successfully.

Reading entity: payments
File format: json
Columns: 13
DataFrame initialized successfully.

Total DataFrames initialized: 5
ALL SOURCE DATAFRAMES INITIALIZED SUCCESSFULLY


In [0]:
print("=" * 60)
print("SOURCE RECORD COUNT VALIDATION")
print("=" * 60)

record_count_results = []

for entity, df in source_dataframes.items():

    expected_count = SOURCE_CONFIG[entity]["expected_count"]

    actual_count = df.count()

    difference = actual_count - expected_count

    status = (
        "PASSED"
        if actual_count == expected_count
        else "FAILED"
    )

    record_count_results.append({
        "entity": entity,
        "expected_count": expected_count,
        "actual_count": actual_count,
        "difference": difference,
        "status": status
    })

    print(f"\nEntity: {entity}")
    print(f"Expected records: {expected_count:,}")
    print(f"Actual records: {actual_count:,}")
    print(f"Difference: {difference:,}")
    print(f"Status: {status}")


total_expected = sum(
    result["expected_count"]
    for result in record_count_results
)

total_actual = sum(
    result["actual_count"]
    for result in record_count_results
)

all_counts_valid = all(
    result["status"] == "PASSED"
    for result in record_count_results
)

print("\n" + "=" * 60)
print(f"Total expected records: {total_expected:,}")
print(f"Total actual records: {total_actual:,}")
print(f"Total difference: {total_actual - total_expected:,}")

if all_counts_valid:
    print("\nALL RECORD COUNT VALIDATIONS PASSED")
else:
    raise RuntimeError("Source record count validation failed.")

SOURCE RECORD COUNT VALIDATION

Entity: customers
Expected records: 1,000
Actual records: 1,000
Difference: 0
Status: PASSED

Entity: products
Expected records: 200
Actual records: 200
Difference: 0
Status: PASSED

Entity: orders
Expected records: 5,000
Actual records: 5,000
Difference: 0
Status: PASSED

Entity: order_items
Expected records: 10,055
Actual records: 10,055
Difference: 0
Status: PASSED

Entity: payments
Expected records: 5,000
Actual records: 5,000
Difference: 0
Status: PASSED

Total expected records: 21,255
Total actual records: 21,255
Total difference: 0

ALL RECORD COUNT VALIDATIONS PASSED


In [0]:
print("=" * 60)
print("SOURCE SCHEMA INSPECTION")
print("=" * 60)

schema_validation_results = []

expected_columns = {
    "customers": 13,
    "products": 12,
    "orders": 11,
    "order_items": 15,
    "payments": 13
}

for entity, df in source_dataframes.items():

    print("\n" + "-" * 60)
    print(f"Entity: {entity.upper()}")
    print(f"Format: {SOURCE_CONFIG[entity]['format']}")
    print("-" * 60)

    df.printSchema()

    actual_columns = len(df.columns)

    status = (
        "PASSED"
        if actual_columns == expected_columns[entity]
        else "FAILED"
    )

    schema_validation_results.append({
        "entity": entity,
        "expected_columns": expected_columns[entity],
        "actual_columns": actual_columns,
        "status": status
    })

    print(f"Expected columns: {expected_columns[entity]}")
    print(f"Actual columns: {actual_columns}")
    print(f"Column count validation: {status}")


all_schemas_valid = all(
    result["status"] == "PASSED"
    for result in schema_validation_results
)

print("\n" + "=" * 60)

if all_schemas_valid:
    print("ALL COLUMN COUNT VALIDATIONS PASSED")
    print("SOURCE SCHEMA INSPECTION COMPLETED")
else:
    raise RuntimeError("Source column count validation failed.")

SOURCE SCHEMA INSPECTION

------------------------------------------------------------
Entity: CUSTOMERS
Format: csv
------------------------------------------------------------
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- updated_at: string (nullable = true)

Expected columns: 13
Actual columns: 13
Column count validation: PASSED

------------------------------------------------------------
Entity: PRODUCTS
Format: csv
------------------------------------------------------------
root
 |-- product_id: string (nullable = true)
 |-- sku: string 

In [0]:
print("=" * 60)
print("SOURCE DATA PREVIEW")
print("=" * 60)

preview_results = []

for entity, df in source_dataframes.items():

    print("\n" + "-" * 60)
    print(f"Entity: {entity.upper()}")
    print(f"Format: {SOURCE_CONFIG[entity]['format']}")
    print("-" * 60)

    preview_df = df.limit(5)

    display(preview_df)

    preview_results.append({
        "entity": entity,
        "status": "PASSED"
    })

    print(f"{entity} preview completed successfully.")


print("\n" + "=" * 60)
print("ALL SOURCE DATA PREVIEWS COMPLETED")

SOURCE DATA PREVIEW

------------------------------------------------------------
Entity: CUSTOMERS
Format: csv
------------------------------------------------------------


customer_id,first_name,last_name,email,phone,city,state,country,postal_code,customer_segment,is_active,created_at,updated_at
CUST-0000001,Daniel,Figueroa,customer0000001@example.com,232.318.1960,East William,Alaska,USA,74064,Regular,True,2025-09-20T07:33:32+00:00,2025-09-20T07:33:32+00:00
CUST-0000002,Jennifer,Miles,customer0000002@example.com,(337)894-0265x42351,West Melanieview,Arizona,USA,96148,Regular,True,2023-11-05T21:12:17+00:00,2023-11-05T21:12:17+00:00
CUST-0000003,Amanda,Dudley,customer0000003@example.com,(518)349-5931x0341,Ramirezstad,Hawaii,USA,49021,Premium,True,2024-12-14T16:36:03+00:00,2024-12-14T16:36:03+00:00
CUST-0000004,Jason,Adams,customer0000004@example.com,+1-519-228-3276x48350,Richardland,Indiana,USA,74842,Premium,True,2024-07-05T21:27:06+00:00,2024-07-05T21:27:06+00:00
CUST-0000005,Jason,Porter,customer0000005@example.com,+1-772-542-3884x9696,New Trevor,Wisconsin,USA,06676,Regular,True,2024-05-03T20:02:30+00:00,2024-05-03T20:02:30+00:00


customers preview completed successfully.

------------------------------------------------------------
Entity: PRODUCTS
Format: csv
------------------------------------------------------------


product_id,sku,product_name,category,brand,unit_price,cost_price,currency,stock_quantity,is_active,created_at,updated_at
PROD-0000001,SKU-0000001,Apex Headphones,Electronics,Apex,1919.62,1051.86,USD,715,True,2023-03-01T21:19:00+00:00,2023-03-01T21:19:00+00:00
PROD-0000002,SKU-0000002,Everest Sunscreen,Beauty,Everest,93.3,56.51,USD,590,True,2024-03-20T09:59:28+00:00,2024-03-20T09:59:28+00:00
PROD-0000003,SKU-0000003,Elite Running Shoes,Sports,Elite,78.71,58.27,USD,839,True,2025-12-17T13:07:26+00:00,2025-12-17T13:07:26+00:00
PROD-0000004,SKU-0000004,Apex T-Shirt,Fashion,Apex,132.98,79.57,USD,790,True,2023-08-12T14:13:09+00:00,2023-08-12T14:13:09+00:00
PROD-0000005,SKU-0000005,Apex Football,Sports,Apex,157.0,65.87,USD,62,True,2024-12-19T12:27:08+00:00,2024-12-19T12:27:08+00:00


products preview completed successfully.

------------------------------------------------------------
Entity: ORDERS
Format: json
------------------------------------------------------------


created_at,currency,customer_id,order_date,order_id,order_status,sales_channel,shipping_city,shipping_country,shipping_state,updated_at
2024-05-01T16:44:06+00:00,USD,CUST-0000419,2024-05-01T16:44:06+00:00,ORD-000000001,Delivered,Website,Port Raymond,USA,Iowa,2024-05-01T16:44:06+00:00
2025-05-30T20:52:42+00:00,USD,CUST-0000030,2025-05-30T20:52:42+00:00,ORD-000000002,Delivered,Website,Port Andrew,USA,Georgia,2025-05-30T20:52:42+00:00
2025-06-03T21:02:31+00:00,USD,CUST-0000161,2025-06-03T21:02:31+00:00,ORD-000000003,Delivered,Website,New Stacy,USA,New York,2025-06-03T21:02:31+00:00
2025-06-03T06:53:43+00:00,USD,CUST-0000650,2025-06-03T06:53:43+00:00,ORD-000000004,Delivered,Mobile App,West Jessicamouth,USA,Arizona,2025-06-03T06:53:43+00:00
2025-01-31T20:52:21+00:00,USD,CUST-0000687,2025-01-31T20:52:21+00:00,ORD-000000005,Delivered,Website,New Wendy,USA,New Mexico,2025-01-31T20:52:21+00:00


orders preview completed successfully.

------------------------------------------------------------
Entity: ORDER_ITEMS
Format: parquet
------------------------------------------------------------


order_item_id,order_id,product_id,quantity,unit_price,cost_price,discount_percentage,gross_amount,discount_amount,net_amount,total_cost,profit_amount,currency,created_at,updated_at
ITEM-0000000001,ORD-000000001,PROD-0000145,1,31.34,15.55,0,31.34,0.00,31.34,15.55,15.79,USD,2024-05-01T16:44:06+00:00,2024-05-01T16:44:06+00:00
ITEM-0000000002,ORD-000000002,PROD-0000160,1,26.39,16.49,0,26.39,0.00,26.39,16.49,9.90,USD,2025-05-30T20:52:42+00:00,2025-05-30T20:52:42+00:00
ITEM-0000000003,ORD-000000003,PROD-0000109,3,165.66,91.27,5,496.98,24.85,472.13,273.81,198.32,USD,2025-06-03T21:02:31+00:00,2025-06-03T21:02:31+00:00
ITEM-0000000004,ORD-000000004,PROD-0000025,2,968.17,749.74,10,1936.34,193.63,1742.71,1499.48,243.23,USD,2025-06-03T06:53:43+00:00,2025-06-03T06:53:43+00:00
ITEM-0000000005,ORD-000000004,PROD-0000045,1,19.42,7.97,10,19.42,1.94,17.48,7.97,9.51,USD,2025-06-03T06:53:43+00:00,2025-06-03T06:53:43+00:00


order_items preview completed successfully.

------------------------------------------------------------
Entity: PAYMENTS
Format: json
------------------------------------------------------------


created_at,currency,net_payment_amount,order_amount,order_id,paid_amount,payment_date,payment_id,payment_method,payment_status,refund_amount,refund_date,updated_at
2024-05-01T16:47:31+00:00,USD,31.34,31.34,ORD-000000001,31.34,2024-05-01T16:47:31+00:00,PAY-000000001,Digital Wallet,Completed,0.00,NaN,2024-05-01T16:47:31+00:00
2025-05-30T20:54:40+00:00,USD,26.39,26.39,ORD-000000002,26.39,2025-05-30T20:54:40+00:00,PAY-000000002,Credit Card,Completed,0.00,NaN,2025-05-30T20:54:40+00:00
2025-06-03T21:03:49+00:00,USD,472.13,472.13,ORD-000000003,472.13,2025-06-03T21:03:49+00:00,PAY-000000003,Debit Card,Completed,0.00,NaN,2025-06-03T21:03:49+00:00
2025-06-03T06:54:00+00:00,USD,4498.33,4498.33,ORD-000000004,4498.33,2025-06-03T06:54:00+00:00,PAY-000000004,Debit Card,Completed,0.00,NaN,2025-06-03T06:54:00+00:00
2025-01-31T20:55:05+00:00,USD,997.74,997.74,ORD-000000005,997.74,2025-01-31T20:55:05+00:00,PAY-000000005,Credit Card,Completed,0.00,NaN,2025-01-31T20:55:05+00:00


payments preview completed successfully.

ALL SOURCE DATA PREVIEWS COMPLETED


In [0]:
print("=" * 70)
print("COMMERCELAKE - FINAL SOURCE VALIDATION SUMMARY")
print("=" * 70)

summary_records = []

for entity in SOURCE_CONFIG:

    file_result = next(
        r for r in file_validation_results
        if r["entity"] == entity
    )

    count_result = next(
        r for r in record_count_results
        if r["entity"] == entity
    )

    schema_result = next(
        r for r in schema_validation_results
        if r["entity"] == entity
    )

    preview_result = next(
        r for r in preview_results
        if r["entity"] == entity
    )

    overall_status = (
        "PASSED"
        if all([
            file_result["status"] == "PASSED",
            file_result["file_size_bytes"] > 0,
            count_result["status"] == "PASSED",
            schema_result["status"] == "PASSED",
            preview_result["status"] == "PASSED"
        ])
        else "FAILED"
    )

    summary_records.append({
        "entity": entity,
        "file_format": SOURCE_CONFIG[entity]["format"],
        "batch_id": BATCH_ID,
        "file_size_bytes": file_result["file_size_bytes"],
        "expected_records": count_result["expected_count"],
        "actual_records": count_result["actual_count"],
        "file_validation": file_result["status"],
        "count_validation": count_result["status"],
        "column_count_validation": schema_result["status"],
        "preview_execution": preview_result["status"],
        "overall_status": overall_status
    })


summary_df = spark.createDataFrame(summary_records)

display(summary_df)

total_datasets = len(summary_records)

passed_datasets = sum(
    r["overall_status"] == "PASSED"
    for r in summary_records
)

failed_datasets = total_datasets - passed_datasets

total_records = sum(
    r["actual_records"]
    for r in summary_records
)

total_bytes = sum(
    r["file_size_bytes"]
    for r in summary_records
)

success_rate = (passed_datasets / total_datasets) * 100

print("\n" + "=" * 70)
print("VALIDATION METRICS")
print("=" * 70)

print(f"Batch ID: {BATCH_ID}")
print(f"Total datasets: {total_datasets}")
print(f"Passed datasets: {passed_datasets}")
print(f"Failed datasets: {failed_datasets}")
print(f"Total source records: {total_records:,}")
print(f"Total source size: {total_bytes:,} bytes")
print(f"Validation success rate: {success_rate:.2f}%")

if failed_datasets == 0:
    print("\nALL SOURCE VALIDATIONS PASSED")
    print("SOURCE DATA IS READY FOR BRONZE INGESTION")
else:
    raise RuntimeError("Source validation failed.")

COMMERCELAKE - FINAL SOURCE VALIDATION SUMMARY


actual_records,batch_id,column_count_validation,count_validation,entity,expected_records,file_format,file_size_bytes,file_validation,overall_status,preview_execution
1000,batch_001,PASSED,PASSED,customers,1000,csv,171044,PASSED,PASSED,PASSED
200,batch_001,PASSED,PASSED,products,200,csv,27458,PASSED,PASSED,PASSED
5000,batch_001,PASSED,PASSED,orders,5000,json,1685858,PASSED,PASSED,PASSED
10055,batch_001,PASSED,PASSED,order_items,10055,parquet,391937,PASSED,PASSED,PASSED
5000,batch_001,PASSED,PASSED,payments,5000,json,2030395,PASSED,PASSED,PASSED



VALIDATION METRICS
Batch ID: batch_001
Total datasets: 5
Passed datasets: 5
Failed datasets: 0
Total source records: 21,255
Total source size: 4,306,692 bytes
Validation success rate: 100.00%

ALL SOURCE VALIDATIONS PASSED
SOURCE DATA IS READY FOR BRONZE INGESTION
